# Preparing ABR Imagery 2021 Data for Analysis

**Will be focusing on SO Detail History View**

In [1]:
#import packages
import pandas as pd
import numpy as np
from datetime import datetime
from os import devnull
import xlrd

In [2]:
#since SO Detail History View is in .Xls format, will need to use devnull and xlrd to open
wb = xlrd.open_workbook('SO Detail History View 2021.Xls', logfile = open(devnull, 'w'))
data = pd.read_excel(wb, thousands = ',')

## **Filtering to keep orders MADE in 2021**

In [3]:
#making transaction date and Actu Ship Date  datetime objects
data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], format = "%m/%d/%Y")
data['Act Ship Date'] = pd.to_datetime(data['Act Ship Date'], format = "%m/%d/%Y")

In [4]:
#filtering data made in 2021
data= data[(data['Transaction Date'] >= '2021-01-01') & (data['Transaction Date'] <= '2021-12-31')]

In [5]:
#testing to see that it worked, great it does!
print(data['Transaction Date'].min())
print(data['Transaction Date'].max())

2021-01-01 00:00:00
2021-12-01 00:00:00


# Changing type of Transaction ID to string and formating to match Traverse

In [6]:
data['Transaction ID'] = data['Transaction ID'].astype({'Transaction ID' : 'float'})
data['Transaction ID'] = data['Transaction ID'].astype({'Transaction ID' : 'int'})
data['Transaction ID'] = data['Transaction ID'].astype({'Transaction ID' : str})

In [7]:
data['Transaction ID'] = data['Transaction ID'].apply(lambda x: x.zfill(8))

In [8]:
data.head()
#sucessful

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Item ID,Location ID,Price Code,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-100.79
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-620.62
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-74.09
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-400.04
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-128.90


## Filtering out Customer ID's that are not actually customers

In [9]:
customer_id_filter = ['edr0001','plt0001','Jos0118','Mag0003', 'PDK0001']
data = data[~data['Customer ID'].isin(customer_id_filter)]

In [10]:
data.head()

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Item ID,Location ID,Price Code,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-100.79
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-620.62
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-74.09
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-400.04
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-128.90


## Saving unique items to text file to view them for any items that should't be there

In [11]:
np.savetxt('unique items 2021.txt', data['Item ID'].unique(), fmt='%s')

**All the items look good**

# There are many items with no Price Code, there is another alternative to use called Product Line. Will add this to the table replacing Price Code. Will check Product Line's and make sure all items have Product Lines

**In Traverse, got a list of all the items called 'IN Item View ALL.Xls'. This has the column called Product Line that we want. I will load this in and join with data on 'Item ID'**

In [12]:
#loading items
wb = xlrd.open_workbook('IN Item View ALL.Xls', logfile = open(devnull, 'w'))
items = pd.read_excel(wb)

In [13]:
#filtering to have only item id and prodcut lin
items = items[['Item ID', 'Product Line', 'Additional Description']]

In [14]:
#joining items with data
data = data.merge(items, on = 'Item ID', how = 'left')

In [15]:
#removing Price code
del data['Price Code']



In [16]:
#reordering dataframe
data = data[['History Type','Transaction ID','Customer ID','Transaction Date','Act Ship Date','Transaction Type', 'Description','Additional Description','Item ID',
             'Location ID', 'Product Line','Qty Ship Sell', 'Units ','Unit Price Sell', 'Price Adj Pct', 'Price Adj Type', 'Unit Cost Sell', 'Ship Via', 'Amount']]

In [17]:
#renaming units to remove whitespace
data = data.rename(columns = {'Units ': 'Units'})

In [18]:
data.head()

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-100.79
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-620.62
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-74.09
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-400.04
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-128.90


In [19]:
#removing payment History type
data = data[~(data['History Type'] == 'Payment')]

In [20]:
#viewing unique items that have no Product Line
data['Item ID'][(data['History Type'] == 'Line Item') & (data['Product Line'].isna())].unique()   
#why didn't this give me ALL the product lines with nan and history type line item ?

array(['purr-minor kit', 'purr-bravo kit', nan, 'claw11',
       'jc-3m2fd-ac-4.5', '854-203DSK-ADPT', 'Liftgatefee', 'JC-ST14',
       'JC-ST19', 'purr-glass pack', 'BB-46-B-BROWNHONEY', 'sun',
       'ind-3mc-b2', 'PORTABLECCASE2', '0000-BOROBATCH-LOGO-2XL',
       'crucible30lb', '4-2-INLET-CONVERSIONKIT',
       'Crucible-3inchby1.5inch', 'BEGINNERKIT-NORTELREDMAX',
       'Purr-Bravo Kit', 'GORO3D', 'GORO5', 'GOS6W', 'VW1-01680',
       'CLAW15', 'TUNGSTEN PICK 1/8', 'JC-1014B', 'OPAL POLISHED WHITE',
       'BB-03-M-DAVESBLUES', '854-220k', 'ind-14/10f-hw', 'ind-1fc-b2'],
      dtype=object)

### Adding Product Lines to Items with NaN Product Lines

**I have looked in Traverse for these Item ID's and have found their corresponding Product Line's. The only items that needed made Product Lines were crucible pots**

In [21]:
#replacing Prodcut lines
data.loc[(data['Item ID'] == 'LIftgatefee'), 'Product Line'] = 'service'
data.loc[(data['Item ID'] == 'liftgatefee'), 'Product Line'] = 'service'
data.loc[(data['Item ID'] == 'Liftgatefee'), 'Product Line'] = 'service'

purr_list = ['purr-minor kit', 'purr-bravo kit', 'purr-glass pack', 'Purr-Bravo Kit']
data.loc[(data['Item ID'].isin(purr_list)), 'Product Line'] = 'PURR'

hose_list = ['4-2-INLET-CONVERSIONKIT', '854-203DSK-ADPT', '854-220k']
data.loc[(data['Item ID'].isin(hose_list)), 'Product Line'] = 'Hoses & Acce'

bb_list = ['BB-03-M-DAVESBLUES', 'BB-46-B-BROWNHONEY']
data.loc[(data['Item ID'].isin(bb_list)), 'Product Line'] = 'Boro Batch'

data.loc[(data['Item ID'] == 'BEGINNERKIT-NORTELREDMAX'), 'Product Line'] = 'BEGINNER KIT'

claw_list = ['claw11', 'CLAW15']
data.loc[(data['Item ID'].isin(claw_list)), 'Product Line'] = 'CLAWS'

crucible_list = ['crucible30lb', 'Crucible-3inchby1.5inch']
data.loc[(data['Item ID'].isin(crucible_list)), 'Product Line'] = 'CRUCIBLE'

opal_list = ['GORO3D', 'GORO5', 'GOS6W', 'OPAL POLISHED WHITE']
data.loc[(data['Item ID'].isin(opal_list)), 'Product Line'] = 'OPALS'

joints_list = ['ind-14/10f-hw', 'ind-1fc-b2', 'ind-3mc-b2', 'JC-ST14', 'JC-ST19']
data.loc[(data['Item ID'].isin(joints_list)), 'Product Line'] = 'Glass Joints'

bushings_list = ['JC-1014B', 'jc-3m2fd-ac-4.5']
data.loc[(data['Item ID'].isin(bushings_list)), 'Product Line'] = 'Bushings'

data.loc[(data['Item ID'].isin(['PORTABLECCASE2'])), 'Product Line'] = 'CRAFT ITEMS'

data.loc[(data['Item ID'].isin(['TUNGSTEN PICK 1/8'])), 'Product Line'] = 'RAKES/PICKS'

data.loc[(data['Item ID'].isin(['VW1-01680'])), 'Product Line'] = 'HA TOOLS'

data.loc[(data['Item ID'].isin(['0000-BOROBATCH-LOGO-2XL'])), 'Product Line'] = 'T-SHIRTS'
#dropping the row with 'sun' as the item id
data = data.drop(data.loc[data['Item ID'] == 'sun'].index)



In [22]:
#further exploring line items with nan product line
data.loc[(data['Product Line'].isna()), 'History Type'].value_counts()

Freight      4483
Sales Tax     522
Line Item      22
Misc            1
Name: History Type, dtype: int64

In [23]:
data.loc[(data['Product Line'].isna()) & (data['History Type'] == 'Line Item')]

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
825,Line Item,00181515,tom0001,2021-11-26,2021-11-30,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,GND,0.0
4441,Line Item,00181145,ste0474,2021-11-05,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
7030,Line Item,00180835,ian0020,2021-10-20,2021-10-20,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
11342,Line Item,00180338,dou0045,2021-09-23,2021-09-24,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
12181,Line Item,00180247,pau0049,2021-09-17,2021-09-20,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
15821,Line Item,00179869,pau0081,2021-08-26,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
18183,Line Item,00179578,Nor0008,2021-08-12,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
26794,Line Item,00178647,UOP0001,2021-06-17,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
27111,Line Item,00178609,ala0056,2021-06-16,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,GND,0.0
28260,Line Item,00178439,mar1061,2021-06-09,2021-06-11,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0


**Looking at the cell above, everything with practically all NaN is empty rows,  I know this by double checking the orders in Traverse, These rows can be removed.
can remove Trans ID 00178332
can remove index 41715, this is an emtpy line with information on items that were subbed out
so everything from the table above can be removed from the dataset**


In [24]:
remove_index = data.loc[(data['Product Line'].isna()) & (data['History Type'] == 'Line Item')].index

data = data[~data.index.isin(remove_index)]

In [25]:
#no all the line items are dealt with
data.loc[(data['Product Line'].isna()), 'History Type'].value_counts()

Freight      4483
Sales Tax     522
Misc            1
Name: History Type, dtype: int64

In [26]:
#looking at the miscallaneuous item in Traverse, this item needs to stay as it counts to what the customer paid
data.loc[(data['History Type'] == 'Misc')]

#need to keep the Freight and Sales Tax rows as well, this gives important information and these should not have a product line. in the future on this project, I may
# add product lines for these for help on getting totals for these categories, but for now, I will not put a Product Line on these

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
13202,Misc,00180135,sam0076,2021-09-10,2021-09-10,Invoice,NaN,NaN,NaN,B01,NaN,1.0,NaN,645.0,0.0,Amount,0.0,NaN,645.0


In [27]:
#changing dtypes of the data for values that should be numeric
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 57052 entries, 34 to 62614
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   History Type            57052 non-null  object        
 1   Transaction ID          57052 non-null  object        
 2   Customer ID             57052 non-null  object        
 3   Transaction Date        57052 non-null  datetime64[ns]
 4   Act Ship Date           55342 non-null  datetime64[ns]
 5   Transaction Type        57052 non-null  object        
 6   Description             51938 non-null  object        
 7   Additional Description  33385 non-null  object        
 8   Item ID                 52046 non-null  object        
 9   Location ID             57052 non-null  object        
 10  Product Line            52046 non-null  object        
 11  Qty Ship Sell           57052 non-null  float64       
 12  Units                   52046 non-null  objec

In [28]:
data['Qty Ship Sell'] = data['Qty Ship Sell'].astype({'Qty Ship Sell' : float})
data['Unit Price Sell'] = data['Unit Price Sell'].astype({'Unit Price Sell' : float})
data['Amount'] = data['Amount'].astype({'Amount' : float})

# Adding a Drop Ship column to the data.

**Some items are drop ship items, will add a column detailing whether a particular item is a drop ship item or not.  An item will be a drop ship item if "DROP SHIP" exists in the Additional Description**

In [29]:
#setting nan in additional Description to 'No Additional Description'
data['Additional Description'] = data['Additional Description'].fillna('No Additional Description')

In [30]:
data['Additional Description'].str.contains(pat = 'DROP SHIP')

34       False
35       False
36       False
37       False
38       False
         ...  
62610    False
62611    False
62612    False
62613    False
62614    False
Name: Additional Description, Length: 57052, dtype: bool

In [31]:
#adding column for whether an item is Drop ship 
data['Drop Ship'] = 0

In [32]:
#creatiing conditional for Drop ship column, will either be Y or N
data.loc[(data['Additional Description'].str.contains(pat = 'DROP SHIP') == True), 'Drop Ship'] = 'Y'
data.loc[(data['Additional Description'].str.contains(pat = 'DROP SHIP') == False), 'Drop Ship'] = 'N'

# Handling the rest of the NaN data in the dataset

In [33]:
data.isna().sum()

History Type                  0
Transaction ID                0
Customer ID                   0
Transaction Date              0
Act Ship Date              1710
Transaction Type              0
Description                5114
Additional Description        0
Item ID                    5006
Location ID                   0
Product Line               5006
Qty Ship Sell                 0
Units                      5006
Unit Price Sell               0
Price Adj Pct                 0
Price Adj Type                0
Unit Cost Sell                0
Ship Via                  11748
Amount                        0
Drop Ship                     0
dtype: int64

**for items that have NaT for actual shipping date, these are items that did not ship, will keep this NaT**

**Item ID, Product Line, and Units are all columns that do not have items, I will change these to 'No Item ID', 'No Product Line', 'No Units'**

In [34]:
data['Item ID'].fillna('No Item ID', inplace = True)
data['Product Line'].fillna('No Product Line', inplace = True)
data['Units'].fillna('No Units', inplace = True)

In [35]:
#for Ship via, the NaN values are unknown, so I will put 'unknown', this is because I would have to mainly go through over 10,000 entries to see what the shipping was
# in Traverse. And that is ridiculous
data.loc[(data['Ship Via'].isna())]

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount,Drop Ship
34,Line Item,00181674,sas0013,2021-12-01,2021-12-01,Invoice,ex-5,No Additional Description,sublined item,B01,service,1.0,ea,0.00,0.0,Percent,0.00,NaN,0.00,N
35,Line Item,00181674,sas0013,2021-12-01,2021-12-01,Invoice,ex-5,No Additional Description,sublined item,B01,service,1.0,ea,0.00,0.0,Percent,0.00,NaN,0.00,N
36,Freight,00181674,sas0013,2021-12-01,2021-12-01,Invoice,NaN,No Additional Description,No Item ID,B01,No Product Line,1.0,No Units,55.00,0.0,Amount,0.00,NaN,55.00,N
37,Line Item,00181672,sam0161,2021-12-01,2021-12-01,Invoice,"KEVLAR LATHE TAPE 2"" X 1/8""","Roll of Heavy Duty Lathe Tape is 2"" wide and 1...",KTN44P4001200,B01,Lathe Access,1.0,1/2,129.95,0.0,Percent,58.12,NaN,129.95,N
38,Line Item,00181672,sam0161,2021-12-01,2021-12-01,Invoice,"8"" Small Yellow Kevlar Gloves","One Pair of Reversible Yellow 8"" Kevlar Glo...",855-003s,B01,Gloves & Sle,1.0,EA,11.95,0.0,Percent,0.00,NaN,11.95,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62587,Line Item,00174470,die0005,2021-01-02,2021-01-11,Invoice,GG Blue Satin #4,<h4> Item name: </h4>\n\nGreasy Glass Blue Sat...,GG-T-BlueSatin4,B01,Greasy Glass,1.0,LB,120.00,0.0,Amount,72.00,NaN,120.00,N
62588,Line Item,00174470,die0005,2021-01-02,2021-01-11,Invoice,GG Blue Cheese B Grade Tube,<h4> Item name: </h4>\n\nGreasy Glass Blue Che...,GG-T-BlueCheese-B,B01,Greasy Glass,1.0,LB,80.00,0.0,Amount,35.00,NaN,80.00,N
62589,Line Item,00174470,die0005,2021-01-02,2021-01-11,Invoice,Northstar Galaxy Rod (Odds),<h4> Item name: </h4>\n\nNorthstar Galaxy Rod ...,ons152,B01,Northstar Ro,1.0,LB,45.00,0.0,Amount,22.50,NaN,45.00,N
62590,Line Item,00174470,die0005,2021-01-02,2021-01-11,Invoice,Northstar Galaxy Rod,<h4> Item name: </h4>\n\nNorthstar Galaxy Rod ...,ns152,B01,Northstar Ro,2.0,LB,58.50,0.0,Amount,32.50,NaN,117.00,N


In [36]:
#replacing ship via nan with unknown
data['Ship Via'].fillna('Unknown', inplace = True)

In [37]:
#Replacing Description NaN with 'No Description'
data['Description'].fillna('No Description', inplace = True)

In [38]:
#Great, that handles all the NaN data!
data.isna().sum()

History Type                 0
Transaction ID               0
Customer ID                  0
Transaction Date             0
Act Ship Date             1710
Transaction Type             0
Description                  0
Additional Description       0
Item ID                      0
Location ID                  0
Product Line                 0
Qty Ship Sell                0
Units                        0
Unit Price Sell              0
Price Adj Pct                0
Price Adj Type               0
Unit Cost Sell               0
Ship Via                     0
Amount                       0
Drop Ship                    0
dtype: int64

In [39]:
data

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount,Drop Ship
34,Line Item,00181674,sas0013,2021-12-01,2021-12-01,Invoice,ex-5,No Additional Description,sublined item,B01,service,1.0,ea,0.00,0.0,Percent,0.000,Unknown,0.00,N
35,Line Item,00181674,sas0013,2021-12-01,2021-12-01,Invoice,ex-5,No Additional Description,sublined item,B01,service,1.0,ea,0.00,0.0,Percent,0.000,Unknown,0.00,N
36,Freight,00181674,sas0013,2021-12-01,2021-12-01,Invoice,No Description,No Additional Description,No Item ID,B01,No Product Line,1.0,No Units,55.00,0.0,Amount,0.000,Unknown,55.00,N
37,Line Item,00181672,sam0161,2021-12-01,2021-12-01,Invoice,"KEVLAR LATHE TAPE 2"" X 1/8""","Roll of Heavy Duty Lathe Tape is 2"" wide and 1...",KTN44P4001200,B01,Lathe Access,1.0,1/2,129.95,0.0,Percent,58.120,Unknown,129.95,N
38,Line Item,00181672,sam0161,2021-12-01,2021-12-01,Invoice,"8"" Small Yellow Kevlar Gloves","One Pair of Reversible Yellow 8"" Kevlar Glo...",855-003s,B01,Gloves & Sle,1.0,EA,11.95,0.0,Percent,0.000,Unknown,11.95,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62610,Line Item,00174473,ada0153,2021-01-02,2021-01-11,Invoice,Carlisle Bunsen #122 - Four-Barrel,<h4> Item name: </h4>\n\nCarlisle Bunsen #122 ...,bunsen122,B01,CARLISLE,1.0,EA,139.95,0.0,Amount,105.000,GND,139.95,N
62611,Line Item,00174473,ada0153,2021-01-02,2021-01-11,Invoice,Y-Splitter w/ Valves (Oxygen),"""Y"" Connection with valve - Oxygen _ ""B"" ma...",Y0V,B01,Hoses & Acce,1.0,EA,39.95,0.0,Amount,20.908,GND,39.95,N
62612,Line Item,00174473,ada0153,2021-01-02,2021-01-11,Invoice,"1/4"" ID T-Grade Hose by the Foot",Propane and oxygen T-Grade Twin Line Hose I.D....,hosft,B01,Hoses & Acce,6.0,FT,2.25,0.0,Amount,1.160,GND,13.50,N
62613,Line Item,00174473,ada0153,2021-01-02,2021-01-11,Invoice,Y-Splitter w/ Valves (Propane),Propane Y with Valves on Each Side: for use...,YPV,B01,Hoses & Acce,1.0,EA,39.95,0.0,Amount,20.908,GND,39.95,N


In [40]:
#restructuring data
data = data[['History Type','Transaction ID','Customer ID','Transaction Date','Act Ship Date','Transaction Type', 'Description','Additional Description','Item ID', 'Drop Ship',
             'Location ID', 'Product Line','Qty Ship Sell', 'Units','Unit Price Sell', 'Price Adj Pct', 'Price Adj Type', 'Unit Cost Sell', 'Ship Via', 'Amount']]

In [41]:
#resetting index
data.reset_index(drop = True, inplace = True)

# Adding geographic information to orders

**I have a file of all the customers and their address information called 'AR Customer View ALL.Xls'. I will join this with my data and get State and Country**

In [42]:
#importing Customer address
wb = xlrd.open_workbook('AR Customer View ALL.Xls', logfile = open(devnull, 'w'))
customer_data = pd.read_excel(wb)

In [43]:
#only need Customer ID, city, region and country
customer_data = customer_data[['Customer ID', 'City', 'Region', 'Country']]

In [44]:
#joining customer data with data
data = data.merge(customer_data, on = 'Customer ID', how = 'left')

In [45]:
data = data[['History Type','Transaction ID','Customer ID', 'City', 'Region', 'Country', 'Transaction Date','Act Ship Date','Transaction Type', 'Description',
             'Additional Description','Item ID', 'Drop Ship','Location ID', 'Product Line','Qty Ship Sell', 'Units','Unit Price Sell', 'Price Adj Pct', 'Price Adj Type',
             'Unit Cost Sell', 'Ship Via', 'Amount']]

In [46]:
data.head()

,History Type,Transaction ID,Customer ID,City,Region,Country,Transaction Date,Act Ship Date,Transaction Type,Description,...,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Line Item,00181674,sas0013,Los Angeles,CA,USA,2021-12-01,2021-12-01,Invoice,ex-5,...,B01,service,1.0,ea,0.00,0.0,Percent,0.00,Unknown,0.00
1,Line Item,00181674,sas0013,Los Angeles,CA,USA,2021-12-01,2021-12-01,Invoice,ex-5,...,B01,service,1.0,ea,0.00,0.0,Percent,0.00,Unknown,0.00
2,Freight,00181674,sas0013,Los Angeles,CA,USA,2021-12-01,2021-12-01,Invoice,No Description,...,B01,No Product Line,1.0,No Units,55.00,0.0,Amount,0.00,Unknown,55.00
3,Line Item,00181672,sam0161,Hammond,IN,USA,2021-12-01,2021-12-01,Invoice,"KEVLAR LATHE TAPE 2"" X 1/8""",...,B01,Lathe Access,1.0,1/2,129.95,0.0,Percent,58.12,Unknown,129.95
4,Line Item,00181672,sam0161,Hammond,IN,USA,2021-12-01,2021-12-01,Invoice,"8"" Small Yellow Kevlar Gloves",...,B01,Gloves & Sle,1.0,EA,11.95,0.0,Percent,0.00,Unknown,11.95


## Unit Cost Sell is a disaster.

## There are items with no unit cost sell, items with unit cost sell above unit price sell, and items with unit cost sell = unit price sell, this data is not useable in its current state. 

## I could try to do a reasonable estimation on the unit cost sell, but it would be just that, an estimation. I may do this in the future, but for now I want to work with data that has been validated and known to be correct, not guesses

In [47]:
data.loc[(data['Unit Cost Sell'] >= data['Unit Price Sell'])].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11492 entries, 0 to 57046
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   History Type            11492 non-null  object        
 1   Transaction ID          11492 non-null  object        
 2   Customer ID             11492 non-null  object        
 3   City                    11474 non-null  object        
 4   Region                  11254 non-null  object        
 5   Country                 11492 non-null  object        
 6   Transaction Date        11492 non-null  datetime64[ns]
 7   Act Ship Date           11016 non-null  datetime64[ns]
 8   Transaction Type        11492 non-null  object        
 9   Description             11492 non-null  object        
 10  Additional Description  11492 non-null  object        
 11  Item ID                 11492 non-null  object        
 12  Drop Ship               11492 non-null  object

**removing unit cost sell from data. Can also remove Description and Additional Description as I have gotten the information that I needed from these columns**

In [48]:
del data['Unit Cost Sell']
del data['Description']
del data['Additional Description']

In [49]:
data

,History Type,Transaction ID,Customer ID,City,Region,Country,Transaction Date,Act Ship Date,Transaction Type,Item ID,Drop Ship,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Ship Via,Amount
0,Line Item,00181674,sas0013,Los Angeles,CA,USA,2021-12-01,2021-12-01,Invoice,sublined item,N,B01,service,1.0,ea,0.00,0.0,Percent,Unknown,0.00
1,Line Item,00181674,sas0013,Los Angeles,CA,USA,2021-12-01,2021-12-01,Invoice,sublined item,N,B01,service,1.0,ea,0.00,0.0,Percent,Unknown,0.00
2,Freight,00181674,sas0013,Los Angeles,CA,USA,2021-12-01,2021-12-01,Invoice,No Item ID,N,B01,No Product Line,1.0,No Units,55.00,0.0,Amount,Unknown,55.00
3,Line Item,00181672,sam0161,Hammond,IN,USA,2021-12-01,2021-12-01,Invoice,KTN44P4001200,N,B01,Lathe Access,1.0,1/2,129.95,0.0,Percent,Unknown,129.95
4,Line Item,00181672,sam0161,Hammond,IN,USA,2021-12-01,2021-12-01,Invoice,855-003s,N,B01,Gloves & Sle,1.0,EA,11.95,0.0,Percent,Unknown,11.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57047,Line Item,00174473,ada0153,Colorado Springs,CO,USA,2021-01-02,2021-01-11,Invoice,bunsen122,N,B01,CARLISLE,1.0,EA,139.95,0.0,Amount,GND,139.95
57048,Line Item,00174473,ada0153,Colorado Springs,CO,USA,2021-01-02,2021-01-11,Invoice,Y0V,N,B01,Hoses & Acce,1.0,EA,39.95,0.0,Amount,GND,39.95
57049,Line Item,00174473,ada0153,Colorado Springs,CO,USA,2021-01-02,2021-01-11,Invoice,hosft,N,B01,Hoses & Acce,6.0,FT,2.25,0.0,Amount,GND,13.50
57050,Line Item,00174473,ada0153,Colorado Springs,CO,USA,2021-01-02,2021-01-11,Invoice,YPV,N,B01,Hoses & Acce,1.0,EA,39.95,0.0,Amount,GND,39.95


## Looking at Address data that is NaN

In [54]:
data.isna().sum()

History Type           0
Transaction ID         0
Customer ID            0
City                  99
Region              1620
Country                0
Transaction Date       0
Act Ship Date       1710
Transaction Type       0
Item ID                0
Drop Ship              0
Location ID            0
Product Line           0
Qty Ship Sell          0
Units                  0
Unit Price Sell        0
Price Adj Pct          0
Price Adj Type         0
Ship Via               0
Amount                 0
dtype: int64

In [55]:
#so every country is filled. that is good, for NaN cities and Regions, I will keep NaN. I will be using Tableau for data visualization, so I believe these need to be NaN for 
#the spatial graphs to work

## Great! I believe this data is ready for Visualization. will write to a file

In [57]:
data.to_csv('ABR Imagery 2021 Orders.csv', index = False)